In [1]:
from delta import configure_spark_with_delta_pip, DeltaTable
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

get_ipython().run_line_magic('load_ext', 'sparksql_magic')
get_ipython().run_line_magic('config', 'SparkSql.limit=20')

builder = (SparkSession.builder
           .appName("join-data")
           .master("spark://spark-master:7077")
           .config("spark.executor.memory", "512m")
           .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
           .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog"))

spark = configure_spark_with_delta_pip(builder, ['org.apache.spark:spark-sql-kafka-0-10_2.12:3.4.1']).getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

:: loading settings :: url = jar:file:/usr/local/lib/python3.12/dist-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /root/.ivy2/cache
The jars for the packages stored in: /root/.ivy2/jars
io.delta#delta-core_2.12 added as a dependency
org.apache.spark#spark-sql-kafka-0-10_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-830ceeed-ecdf-4b5b-9a75-c7ebd8b1efda;1.0
	confs: [default]
	found io.delta#delta-core_2.12;2.4.0 in central
	found io.delta#delta-storage;2.4.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
	found org.apache.spark#spark-sql-kafka-0-10_2.12;3.4.1 in central
	found org.apache.spark#spark-token-provider-kafka-0-10_2.12;3.4.1 in central
	found org.apache.kafka#kafka-clients;3.3.2 in central
	found org.lz4#lz4-java;1.8.0 in central
	found org.xerial.snappy#snappy-java;1.1.10.1 in central
	found org.slf4j#slf4j-api;2.0.6 in central
	found org.apache.hadoop#hadoop-client-runtime;3.3.4 in central
	found org.apache.hadoop#hadoop-client-api;3.3.4 in central
	found commons-logging#commons-logging;1.1.3 in centra

In [2]:
streaming_schema = StructType([
    StructField("order_id", IntegerType()),
    StructField("product_id", IntegerType()),
    StructField("quantity", IntegerType()),
    StructField("timestamp", IntegerType())])

streaming_df = (spark.readStream
                .format("kafka")
                .option("kafka.bootstrap.servers", "kafka:9092")
                .option("subscribe", "orders")
                .option("startingOffsets", "earliest")
                .option("failOnDataLoss", "false")
                .load()
                .withColumn('value', from_json(col('value').cast("STRING"), streaming_schema)))

streaming_df = (streaming_df
                .select(
                    col('value.order_id').alias('order_id'),
                    col('value.product_id').alias('product_id'),
                    col('value.quantity').alias('quantity'),
                    to_timestamp(col("timestamp"), "MM/dd/yyyy, HH:mm:ss").alias('timestamp')))


In [3]:
product_details = [
    (1001, "Laptop", 999.99),
    (1002, "Mouse", 19.99),
    (1003, "Keyboard", 29.99),
    (1004, "Monitor", 199.99),
    (1005, "Speaker", 49.99)]

columns = ["product_id", "name", "price"]

static_df = spark.createDataFrame(product_details, columns)
static_df.show()

+----------+--------+------+
|product_id|    name| price|
+----------+--------+------+
|      1001|  Laptop|999.99|
|      1002|   Mouse| 19.99|
|      1003|Keyboard| 29.99|
|      1004| Monitor|199.99|
|      1005| Speaker| 49.99|
+----------+--------+------+



In [5]:
joined_df = (streaming_df
             .join(static_df, streaming_df.product_id == static_df.product_id, "inner")
             .drop(static_df.product_id)
             .withColumn('invoice_amount', streaming_df.quantity * static_df.price))

In [6]:
query = (joined_df.writeStream
         .format("delta")
         .outputMode("append")
         .option("failureOnDataLoss", "true")
         .option("checkpointLocation", "/opt/workspace/data/delta_lake/joining-stream-static/orders")
         .start("/opt/workspace/data/delta_lake/joining-stream-static/orders"))

In [8]:
%%sparksql

SELECT * FROM delta.`/opt/workspace/data/delta_lake/joining-stream-static/orders`;

only showing top 20 row(s)


order_id,product_id,quantity,timestamp,name,price,invoice_amount
360689,1004,3,2025-06-11 08:54:11.074000,Monitor,199.99,599.97
205462,1004,4,2025-06-11 08:54:21.076000,Monitor,199.99,799.96
181267,1004,3,2025-06-11 08:54:41.077000,Monitor,199.99,599.97
498144,1003,4,2025-06-11 08:55:21.082000,Keyboard,29.99,119.96
716091,1004,5,2025-06-11 08:58:21.096000,Monitor,199.99,999.95
266822,1005,4,2025-06-11 08:55:31.084000,Speaker,49.99,199.96
326957,1005,3,2025-06-11 08:56:41.089000,Speaker,49.99,149.97
100199,1005,2,2025-06-11 08:57:31.092000,Speaker,49.99,99.98
989293,1004,1,2025-06-11 08:56:51.089000,Monitor,199.99,199.99
825175,1005,3,2025-06-11 08:58:11.095000,Speaker,49.99,149.97


25/06/11 08:58:47 ERROR TaskSetManager: Task 0 in stage 158.0 failed 4 times; aborting job


In [9]:
query.stop()

In [10]:
spark.stop()